# Gradient Boosting — hausse du loyer effectif en 2026

Ce notebook utilise `df_final` préparé dans `model_testing.ipynb`. La cible est `effective_growth_pct` : variation annualisée du loyer effectif des mêmes appartements. Renouvellements et relocations sont inclus.

Le pipeline encode le marché et le type de bail et entraîne un `GradientBoostingRegressor`. Les variables publiques sont décalées d'un an; le loyer effectif courant et les concessions du nouveau bail sont exclus pour éviter d'utiliser des informations liées à la cible.

Le test porte sur 2025, après entraînement sur les années précédentes. Un second modèle est ajusté jusqu'en 2025 pour prévoir 2026. Aucun CSV ni fichier de modèle n'est créé.

Un modèle commun est conservé. `Market` et `type_bail` restent des features catégorielles; `bedrooms` est ajouté aux features numériques. Les seuls facteurs publics retenus sont `general_cpi`, `vacancy_rate` et `unemployement_rate`, décalés d’un an.

In [ ]:
%run ./model_testing.ipynb

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGET = "effective_growth_pct"
public_features = ["general_cpi", "vacancy_rate", "unemployement_rate"]
# Repartir des attributs privés de df_final, puis rattacher les facteurs à Year - 1.
regression_data = df_final.drop(columns=public_features).copy()
public_lagged = df_public[["Market", "Year"] + public_features].copy()
public_lagged["public_source_year"] = public_lagged.Year
public_lagged["Year"] = public_lagged.Year + 1
public_lagged = public_lagged.rename(columns={c: "lag1_" + c for c in public_features})
regression_data = regression_data.merge(public_lagged, on=["Market", "Year"], how="left", validate="many_to_one")
NUMERIC_FEATURES = ["prev_effective_rent", "gap_years", "bedrooms"] + ["lag1_" + c for c in public_features]
CATEGORICAL_FEATURES = ["Market", "type_bail"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
eligible = (regression_data[FEATURES + [TARGET]].notna().all(axis=1)
            & np.isfinite(regression_data[NUMERIC_FEATURES + [TARGET]].astype(float)).all(axis=1))
model_data = regression_data.loc[eligible].copy()
print("Paires disponibles :", len(regression_data), "; paires utilisables :", len(model_data))
print("Les paires sans facteurs de l'année précédente sont exclues, pas imputées à zéro.")
display(model_data.groupby(["Market", "Year"]).size().rename("nombre_paires").to_frame())
TEST_YEAR = 2025
train = model_data.loc[model_data.Year.lt(TEST_YEAR)].copy()
test = model_data.loc[model_data.Year.eq(TEST_YEAR)].copy()
assert not train.empty and not test.empty
assert train.Year.max() < test.Year.min()
assert set(test.Market).issubset(set(train.Market))

# Ne pas inclure Year : le modèle utilise les facteurs économiques et le marché.
def make_gradient_boosting_model():
    preprocessing = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
        ("category", OneHotEncoder(categories=[MARKETS, ["Relocation", "Renouvellement"]], drop="first", handle_unknown="error", sparse_output=False), CATEGORICAL_FEATURES),
    ])
    return Pipeline([("preprocessing", preprocessing), ("regression", GradientBoostingRegressor(n_estimators=200, learning_rate=0.03, max_depth=2, min_samples_leaf=20, loss="squared_error", random_state=42))])

boosting_model = make_gradient_boosting_model()
boosting_model.fit(train[FEATURES], train[TARGET])
train_prediction = boosting_model.predict(train[FEATURES])
test_prediction = boosting_model.predict(test[FEATURES])
# Référence simple calculée sur l'entraînement uniquement.
baseline = np.full(len(test), train[TARGET].mean())
validation_results = pd.DataFrame([
    {"modele": "Gradient Boosting", "MAE_points": mean_absolute_error(test[TARGET], test_prediction),
     "RMSE_points": np.sqrt(mean_squared_error(test[TARGET], test_prediction)), "R2": r2_score(test[TARGET], test_prediction)},
    {"modele": "Moyenne historique", "MAE_points": mean_absolute_error(test[TARGET], baseline),
     "RMSE_points": np.sqrt(mean_squared_error(test[TARGET], baseline)), "R2": r2_score(test[TARGET], baseline)},
]).set_index("modele")
display(validation_results.round(4))
df_predictions = test[["sPropCode", "sUnitCode", "Market", "Year", "type_bail", TARGET]].copy()
df_predictions["predicted_growth_pct"] = test_prediction
df_predictions["error_points"] = df_predictions.predicted_growth_pct - df_predictions[TARGET]
display(df_predictions.head(15).round(3))

## Comparaison sur le même test

Les paramètres sont fixés avant de regarder le test : 200 arbres, taux d’apprentissage 0,03, profondeur maximale 2, au moins 20 observations par feuille. Cela limite la complexité avec ce petit historique. Ils ne sont pas choisis en optimisant les résultats de 2025.

La régression linéaire ci-dessous utilise exactement les mêmes observations et variables pour une comparaison directe. Le pipeline Gradient Boosting ne standardise pas les valeurs numériques : les arbres n'en ont pas besoin.

In [ ]:
linear_comparison = Pipeline([
    ("preprocessing", ColumnTransformer([
        ("numeric", StandardScaler(), NUMERIC_FEATURES),
        ("category", OneHotEncoder(categories=[MARKETS, ["Relocation", "Renouvellement"]], drop="first", handle_unknown="error", sparse_output=False), CATEGORICAL_FEATURES),
    ])),
    ("regression", LinearRegression()),
])
linear_comparison.fit(train[FEATURES], train[TARGET])
linear_prediction = linear_comparison.predict(test[FEATURES])
comparison_results = validation_results.copy()
comparison_results.loc["Régression linéaire"] = [
    mean_absolute_error(test[TARGET], linear_prediction),
    np.sqrt(mean_squared_error(test[TARGET], linear_prediction)),
    r2_score(test[TARGET], linear_prediction),
]
display(comparison_results.round(4))

## Importance des features

Un Gradient Boosting n'a pas de coefficient unique par variable. On présente donc les importances par permutation sur le test 2025 : combien la MAE se dégrade lorsqu'on mélange une feature. Une valeur positive indique que cette feature aidait les prédictions; une valeur proche de zéro ou négative indique peu d'aide mesurable sur ce test.

Les trois facteurs publics restent corrélés et répétés par marché/année. Le mélange de ces valeurs peut créer des combinaisons peu réalistes; les importances ne représentent pas des effets causaux ni une preuve de significativité.

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

permutation = permutation_importance(boosting_model, test[FEATURES], test[TARGET],
                                    scoring="neg_mean_absolute_error", n_repeats=10, random_state=42, n_jobs=1)
df_feature_importance = pd.DataFrame({
    "feature": FEATURES,
    "degradation_MAE_moyenne_points": permutation.importances_mean,
    "ecart_type_repetitions": permutation.importances_std,
}).sort_values("degradation_MAE_moyenne_points", ascending=False)
display(df_feature_importance.round(4))
fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].scatter(test[TARGET], test_prediction, alpha=0.25, s=15)
low=min(test[TARGET].min(),test_prediction.min()); high=max(test[TARGET].max(),test_prediction.max())
axes[0].plot([low,high],[low,high],"--",color="gray")
axes[0].set(title="Gradient Boosting : test 2025",xlabel="Hausse observée (%)",ylabel="Hausse prédite (%)")
importance_plot = df_feature_importance.sort_values("degradation_MAE_moyenne_points")
axes[1].barh(importance_plot.feature, importance_plot.degradation_MAE_moyenne_points,
             xerr=importance_plot.ecart_type_repetitions, alpha=0.8)
axes[1].axvline(0,color="black",linewidth=1)
axes[1].set(title="Importance par permutation sur le test",xlabel="Dégradation de la MAE (points)")
for ax in axes:ax.grid(alpha=0.3)
fig.tight_layout();plt.show()

## Prévision finale 2026

Après la validation sur 2025, nous ajustons un **nouveau modèle** sur toutes les paires admissibles jusqu'en 2025. Le modèle de validation et ses métriques restent distincts.

Hypothèses pour 2026 :

- Les facteurs publics utilisés sont ceux de 2025, puisque le modèle les décale d'un an.
- Le loyer effectif de référence est celui du dernier bail connu de chaque appartement au 31 décembre 2025.
- L'intervalle prévu entre deux baux est fixé à **1 an**. Il s'agit d'un scénario annualisé pour les appartements de l'extrait, pas d'un calendrier réel de renouvellements.
- Nous calculons deux scénarios par appartement : renouvellement et relocation. La part prévue de relocations est estimée avec les paires comparables de 2025 de son marché. Cette part est une hypothèse modifiable.
- Chaque appartement de l'extrait a le même poids. Le résultat est une hausse moyenne prédite dans l'échantillon; il ne mesure ni les recettes ni l'occupation du portefeuille réel.

Le modèle ne prévoit pas explicitement les promotions de 2026 : il prévoit la variation du loyer effectif à partir de l'historique. Ses limites de validation et de stabilité des coefficients restent applicables.

Ce modèle Gradient Boosting fondé sur des arbres n’extrapole pas une tendance linéaire au-delà du domaine historique. La prévision peut donc différer de celle de la régression linéaire lorsque les facteurs publics sortent du domaine historique.

In [ ]:
FORECAST_YEAR = 2026
final_training_data = model_data.loc[model_data.Year.le(2025)].copy()
final_model = make_gradient_boosting_model()
final_model.fit(final_training_data[FEATURES], final_training_data[TARGET])
print(f"Gradient Boosting final : {len(final_training_data):,} paires jusqu'en 2025.")
encoded_names = final_model.named_steps["preprocessing"].get_feature_names_out()
df_final_tree_importance = pd.DataFrame({
    "feature": encoded_names,
    "importance_impurete": final_model.named_steps["regression"].feature_importances_,
}).sort_values("importance_impurete",ascending=False)
# Diagnostic de l'entraînement; l'importance par permutation sur le test est privilégiée.
display(df_final_tree_importance.round(4))

In [ ]:
# Tous les appartements ayant un bail connu, pas uniquement ceux avec une paire récente.
CUTOFF = pd.Timestamp("2025-12-31")
known_leases = private_leases.loc[private_leases.lease_start.le(CUTOFF)].copy()
latest = known_leases.sort_values(UNIT_KEY + ["lease_start"]).groupby(UNIT_KEY, as_index=False).tail(1).copy()
latest["Market"] = latest.sState.map({"Quebec": "Montréal", "Ontario": "Ottawa — partie Ontario"})
latest = latest.loc[latest.effective_rent.gt(0) & np.isfinite(latest.effective_rent) & latest.Market.notna()].copy()
assert not latest.duplicated(UNIT_KEY).any()

# Proportions issues des paires 2025, calculées séparément par marché.
recent = df_final.loc[df_final.Year.eq(2025) & df_final.type_bail.isin(["Relocation", "Renouvellement"])]
relocation_share_2026 = recent.assign(relocation=recent.type_bail.eq("Relocation")).groupby("Market").relocation.mean().to_dict()
print("Hypothèse de part des relocations par marché :", relocation_share_2026)
# Pour modifier l'hypothèse : relocation_share_2026["Montréal"] = 0.30

scenario_parts = []
for kind in ["Renouvellement", "Relocation"]:
    scenario = latest[UNIT_KEY + ["sBuilding", "Market", "effective_rent", "lease_start", "sBeds"]].copy()
    scenario = scenario.rename(columns={"effective_rent": "prev_effective_rent", "lease_start": "reference_lease_start", "sBeds": "bedrooms"})
    scenario["Year"] = FORECAST_YEAR
    scenario["gap_years"] = 1.0
    scenario["type_bail"] = kind
    scenario_parts.append(scenario)
scenarios_2026 = pd.concat(scenario_parts, ignore_index=True)
source_2025 = df_public.loc[df_public.Year.eq(2025), ["Market"] + public_features].rename(columns={c: "lag1_" + c for c in public_features})
scenarios_2026 = scenarios_2026.merge(source_2025, on="Market", how="left", validate="many_to_one")
assert scenarios_2026[FEATURES].notna().all().all()
assert len(scenarios_2026) == 2 * len(latest)
scenarios_2026["predicted_growth_pct"] = final_model.predict(scenarios_2026[FEATURES])
scenarios_2026["predicted_effective_rent"] = scenarios_2026.prev_effective_rent * (
    1 + scenarios_2026.predicted_growth_pct / 100
) ** scenarios_2026.gap_years
assert scenarios_2026.predicted_effective_rent.gt(0).all()
display(scenarios_2026.groupby(["Market", "type_bail"]).agg(
    nombre_appartements=("predicted_growth_pct", "size"),
    hausse_moyenne_pct=("predicted_growth_pct", "mean"),
    hausse_mediane_pct=("predicted_growth_pct", "median"),
).round(3))

In [ ]:
def combine_2026_scenarios(scenarios, relocation_shares):
    result = scenarios.copy()
    shares = result.Market.map(relocation_shares)
    if shares.isna().any() or not shares.between(0, 1).all():
        raise ValueError("Chaque marché doit avoir une proportion de relocations entre 0 et 1.")
    result["scenario_weight"] = np.where(result.type_bail.eq("Relocation"), shares, 1 - shares)
    result["weighted_growth"] = result.predicted_growth_pct * result.scenario_weight
    unit_forecasts = result.groupby(UNIT_KEY + ["Market", "sBuilding"], as_index=False).agg(
        predicted_growth_pct=("weighted_growth", "sum"), total_weight=("scenario_weight", "sum")
    )
    assert np.allclose(unit_forecasts.total_weight, 1)
    return unit_forecasts.drop(columns="total_weight")

def estimate_2026(relocation_shares=None):
    """Hausse effective moyenne prévue en 2026 (%) pour les appartements de l'extrait."""
    shares = relocation_share_2026 if relocation_shares is None else relocation_shares
    return float(combine_2026_scenarios(scenarios_2026, shares).predicted_growth_pct.mean())

df_forecast_2026 = combine_2026_scenarios(scenarios_2026, relocation_share_2026)
forecast_2026_pct = estimate_2026()
print(f"HAUSSE EFFECTIVE MOYENNE PRÉVUE EN 2026 : {forecast_2026_pct:.2f} %")
print(f"Scénario fondé sur {len(df_forecast_2026):,} appartements de l'extrait, pondérés également.")
display(df_forecast_2026.groupby("Market").agg(
    nombre_appartements=("predicted_growth_pct", "size"), hausse_moyenne_pct=("predicted_growth_pct", "mean")
).round(3))
display(df_forecast_2026.head(15).round(3))

# Sensibilité de la moyenne prévue au choix des poids renouvellement/relocation.
sensitivity_2026 = pd.DataFrame([
    {"scenario": "Tous renouvellements", "hausse_pct": estimate_2026({m: 0 for m in relocation_share_2026})},
    {"scenario": "Parts observées en 2025", "hausse_pct": forecast_2026_pct},
    {"scenario": "Toutes relocations", "hausse_pct": estimate_2026({m: 1 for m in relocation_share_2026})},
])
display(sensitivity_2026.round(3))

## Fonctions du gabarit

`estimate_2026(leases, asking, external=None)` refait l'entraînement et retourne la hausse moyenne prévue. `backtest(leases, target_year)` n'utilise que les baux antérieurs à l'année cible pour apprendre. Les loyers affichés `asking` sont acceptés dans la signature mais ne sont pas utilisés dans cette méthode de loyers effectifs.

Les proportions de renouvellement/relocation sont celles de la dernière année de paires utilisables par marché. Les années test ne sont pas utilisées pour sélectionner les paramètres du Gradient Boosting. La disponibilité exacte des publications publiques à chaque date reste à vérifier.

In [ ]:
def prepare_template_leases(leases):
    data = leases.copy()
    if "bedrooms" not in data:
        data["bedrooms"] = pd.to_numeric(data.sBeds, errors="raise")
    data = data.rename(columns={k: v for k, v in LEASE_COLUMN_RENAMES.items() if k in data and v not in data})
    data["lease_start"] = pd.to_datetime(data.lease_start)
    data["lease_start_year"] = data.lease_start.dt.year
    data["Market"] = data.sState.map({"Quebec": "Montréal", "Ontario": "Ottawa — partie Ontario"})
    data["Year"] = data.lease_start_year
    data["type_bail"] = data.is_renewal.map({0: "Relocation", 1: "Renouvellement"})
    return data


def prepare_template_pairs(leases, external):
    pairs = same_unit_growth(leases, "effective_rent")
    pairs[TARGET] = pairs.growth_pct
    public = external[["Market", "Year"] + public_features].copy()
    assert not public.duplicated(["Market", "Year"]).any()
    public["Year"] += 1
    public = public.rename(columns={c: "lag1_" + c for c in public_features})
    pairs = pairs.merge(public, on=["Market", "Year"], how="left", validate="many_to_one")
    return pairs.dropna(subset=FEATURES + [TARGET])


def fit_template_model(leases, year, external):
    history = prepare_template_leases(leases)
    history = history.loc[history.lease_start.lt(pd.Timestamp(year=year, month=1, day=1))].copy()
    public_history = external.loc[external.Year.lt(year)].copy()
    training = prepare_template_pairs(history, public_history)
    if training.empty:
        raise ValueError("Pas de paires et de facteurs publics disponibles avant l'année cible.")
    # Catégories définies à l'avance; un marché sans historique est identifié dans le backtest.
    fitted = make_gradient_boosting_model()
    fitted.fit(training[FEATURES], training[TARGET])
    return fitted, history, training, public_history


def estimate_2026(leases, asking, external=None):
    """Hausse effective annualisée moyenne prévue en 2026 (%), mêmes appartements."""
    public = df_public if external is None else external
    fitted, history, training, public_history = fit_template_model(leases, 2026, public)
    latest = history.sort_values(UNIT_KEY + ["lease_start"]).groupby(UNIT_KEY).tail(1).copy()
    latest = latest.loc[latest.effective_rent.gt(0) & latest.Market.notna()]
    parts = []
    # Part de relocations de la dernière année de paires utilisables dans chaque marché.
    shares = {}
    for market, group in training.groupby("Market"):
        recent = group.loc[group.Year.eq(group.Year.max())]
        shares[market] = recent.type_bail.eq("Relocation").mean()
    for kind in ["Renouvellement", "Relocation"]:
        scenario = latest[UNIT_KEY + ["Market", "effective_rent", "bedrooms"]].rename(columns={"effective_rent": "prev_effective_rent"}).copy()
        scenario["gap_years"] = 1.0
        scenario["type_bail"] = kind
        source = public_history.loc[public_history.Year.eq(2025), ["Market"] + public_features].rename(columns={c: "lag1_" + c for c in public_features})
        scenario = scenario.merge(source, on="Market", how="left", validate="many_to_one")
        if scenario[FEATURES].isna().any().any():
            raise ValueError("Facteurs publics 2025 ou caractéristiques de scénario manquants.")
        share = scenario.Market.map(shares)
        if share.isna().any():
            raise ValueError("Aucun historique pour estimer le poids d'un marché.")
        scenario["weight"] = share if kind == "Relocation" else 1 - share
        scenario["weighted_growth"] = fitted.predict(scenario[FEATURES]) * scenario.weight
        parts.append(scenario)
    unit_growth = pd.concat(parts).groupby(UNIT_KEY).weighted_growth.sum()
    return float(unit_growth.mean())


def backtest(leases, target_year):
    """Entraîne avant target_year et compare aux paires réelles de cette année."""
    fitted, history, training, public_history = fit_template_model(leases, target_year, df_public)
    # Les baux de l'année cible ne servent qu'à mesurer Y après l'entraînement.
    evaluation_leases = prepare_template_leases(leases)
    evaluation_leases = evaluation_leases.loc[evaluation_leases.lease_start.lt(pd.Timestamp(year=target_year + 1, month=1, day=1))]
    evaluation = prepare_template_pairs(evaluation_leases, public_history)
    evaluation = evaluation.loc[evaluation.Year.eq(target_year)]
    if evaluation.empty:
        raise ValueError("Aucune paire évaluable pour l'année cible.")
    predicted = fitted.predict(evaluation[FEATURES])
    unseen_markets = sorted(set(evaluation.Market) - set(training.Market))
    return {
        "annee": target_year, "nombre_paires": len(evaluation),
        "hausse_predite_moyenne_pct": float(predicted.mean()),
        "hausse_observee_moyenne_pct": float(evaluation[TARGET].mean()),
        "erreur_moyenne_points": float(predicted.mean() - evaluation[TARGET].mean()),
        "R2": r2_score(evaluation[TARGET], predicted),
        "MAE_points": mean_absolute_error(evaluation[TARGET], predicted),
        "RMSE_points": np.sqrt(mean_squared_error(evaluation[TARGET], predicted)),
        "marches_sans_historique": ", ".join(unseen_markets),
    }

# Les baux renommés préparés dans model_testing peuvent être passés directement.
asking = pd.read_csv("private_data/equinoxe_asking_history.csv", dtype={"sPropCode": str, "sUnitCode": str})
estimation_gabarit_2026 = estimate_2026(private_leases, asking, external=df_public)
print(f"Estimation 2026 : {estimation_gabarit_2026:.2f} %")
backtest_results = pd.DataFrame([backtest(private_leases, year) for year in (2023, 2024, 2025)])
display(backtest_results.round(4))
assert np.isclose(estimation_gabarit_2026, forecast_2026_pct)


### Résultats disponibles

- `comparison_results` : R², MAE et RMSE du test 2025, Gradient Boosting, régression et référence historique.
- `df_feature_importance` : importances par permutation, une ligne par feature.
- `df_forecast_2026` et `forecast_2026_pct` : prévisions pour l'extrait, sous les hypothèses retenues.
- `backtest_results` : vérifications 2023, 2024 et 2025.

Il n'y a pas encore de métriques d'erreur pour 2026, car les hausses réelles ne figurent pas dans ces données. Le nombre de baux ne remplace pas le petit nombre d'années indépendantes. Les scénarios ne sont pas des prévisions de recettes du portefeuille.